## Step 12 - OCR với Surya v1 (drop-in thay step 6 / 10 / 11)

Notebook này thay engine OCR bằng **Surya v1** (`surya-ocr==0.7.0`) - OCR đa ngôn ngữ của
Datalab, hỗ trợ tiếng Việt có dấu (`vi`) + Anh (`en`). Pipeline chuẩn det → rec:

- `DetectionPredictor` nội bộ (model `vikp/surya_det3`) - phát hiện dòng chữ (bbox)
- `run_ocr` + model rec (`vikp/surya_rec2`) - nhận dạng nội dung từng dòng theo lang

**QUAN TRỌNG - phải pin surya v1**: `pip install surya-ocr` hiện cài "Surya 2" (>=0.8) dùng
`SuryaInferenceManager` tự spawn **vllm trong Docker** → **không chạy được trên Kaggle**.
Notebook này cố định `surya-ocr==0.7.0` (v1, chạy thẳng trên torch/HF).

**Ưu / nhược so với các engine khác trên T4** (1 frame):

| | EasyOCR | Qwen2.5-VL-3B | PP-OCRv5 | Surya v1 |
|---|---|---|---|---|
| Tốc độ | ~0.3 s | ~4.66 s | ~0.05 s | ~0.2–0.5 s (tuỳ độ phân giải) |
| Tiếng Việt có dấu | sai dấu nặng | tốt nhất | tốt | tốt |
| Banner nền phức tạp | hay miss | tốt nhất | khá | khá–tốt |
| VRAM | ~1 GB | ~7 GB | <0.5 GB | ~1–2 GB |
| Cài đặt | easy | transformers | paddle (dễ vướng) | pin surya-ocr==0.7.0 |

**Input / output y hệt step 6 / 10 / 11**:
```
input  : Keyframes/<batch>/<video_name>/<idx>.webp
output : OCR/<video_name>.json   # { "<frame_id>": "<ocr text>", ... }
         ocr_data.json           # merged flat dict, drop-in cho backend
```

**Resumable 2 cấp**: per video (`OCR/<video>.json` đủ frame thì bỏ qua) +
per frame (frame đã có không re-read, checkpoint mỗi `CHECKPOINT_EVERY`).

**Lưu ý**: lần đầu init sẽ tự download model từ Hugging Face (det ~20MB, rec ~1.1GB).
Kaggle có internet nên OK. Nếu gặp lỗi version transformers, đổi pin trong cell install.


In [ ]:
# 1) FIX TORCH/TORCHVISION: pin đúng cặp torch 2.5.1 + torchvision 0.20.1 (cu121).
#    - torch mới hơn (pip nâng lên cu126) -> lỗi `undefined symbol: ncclCommShrink`
#    - torchvision mới hơn (trong image) -> lỗi `torchvision::nms does not exist`
!python -m pip install -q torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121

# 2) Surya v1: `pip install surya-ocr` mới nhất là "Surya 2" (>=0.8) - phải chạy vllm
#    trong Docker -> KHÔNG chạy được trên Kaggle. Dùng surya v1 0.7.0 (chạy thẳng trên
#    torch/HF, không cần Docker) + transformers đúng thời kỳ để tránh API đổi.
!python -m pip install -q "surya-ocr==0.7.0" "transformers==4.46.3"

import torch, torchvision
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
print('torchvision', torchvision.__version__)


In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path


In [ ]:
# === INPUTS - sửa nếu keyframe dataset đặt tên khác =====================
KEYFRAMES_DIR    = '/kaggle/input/aic-keyframes-webp/Keyframes'
OUTPUT_DIR       = './OCR'                 # one <video_name>.json per video (same as step 6)

# === SONG SONG 2 GPU ====================================================
NUM_GPUS         = 2        # số T4 được cấp; đổi thành 1 nếu chỉ có 1 GPU
CONFIG_FILE      = '_surya_config.json'    # config dump cho worker
WORKER_SCRIPT    = '_surya_worker.py'      # script worker, sinh tự động ở cell sau

# === MODEL SURYA v1 (surya-ocr 0.7.0) ===================================
LANGS            = ['vi', 'en']   # Surya hỗ trợ tiếng Việt (vi) + Anh (en)
REC_IMG_HEIGHT   = 256            # độ phân giải crop chữ (height px, width cố định 896).
                                  # Tăng lên 384/512 nếu banner chữ nhỏ bị miss; chậm hơn tương ứng.

# === TUNING ============================================================
FRAME_STRIDE     = 1        # 1 = mọi keyframe; 2 = mỗi 2 frame (giảm 1/2 cost)
IMG_BATCH        = 8        # ảnh / 1 lần gọi run_ocr() - gộp batch để ăn GPU
IMAGE_EXTS       = ('.jpg', '.jpeg', '.png', '.webp')
CHECKPOINT_EVERY = 32       # flush JSON mỗi N frame trong 1 video
MIN_CONF         = 0.0      # lọc text line có confidence < MIN_CONF; 0.0 = giữ hết

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# Mỗi worker init engine 1 lần trên GPU riêng → dump config + sinh worker script.
_config = {
    'min_conf': MIN_CONF,
    'keyframes_dir': str(Path(KEYFRAMES_DIR)),
    'image_batch': IMG_BATCH,
    'image_exts': list(IMAGE_EXTS),
    'output_dir': str(Path(OUTPUT_DIR)),
    'rec_image_height': REC_IMG_HEIGHT,
    'checkpoint_every': CHECKPOINT_EVERY,
    'frame_stride': FRAME_STRIDE,
    'langs': LANGS,
}
with open(CONFIG_FILE, 'w', encoding='utf-8') as f:
    json.dump(_config, f, ensure_ascii=False, indent=2)
print('config ->', CONFIG_FILE)

WORKER_SRC = r'''import argparse
import json
import os
import time
from pathlib import Path

CONFIG_PATH = '_surya_config.json'
with open(CONFIG_PATH, encoding='utf-8') as f:
    cfg = json.load(f)

KEYFRAMES_DIR    = cfg['keyframes_dir']
OUTPUT_DIR       = cfg['output_dir']
FRAME_STRIDE     = cfg['frame_stride']
LANGS            = cfg['langs']
IMG_BATCH        = cfg['image_batch']
IMAGE_EXTS       = tuple(cfg['image_exts'])
CHECKPOINT_EVERY = cfg['checkpoint_every']
MIN_CONF         = cfg['min_conf']
REC_IMG_HEIGHT   = cfg.get('rec_image_height', 256)

os.makedirs(OUTPUT_DIR, exist_ok=True)


def init_engine():
    from surya.settings import settings
    from surya.ocr import run_ocr
    from surya.model.detection.model import load_model as load_det_model
    from surya.model.detection.model import load_processor as load_det_processor
    from surya.model.recognition.model import load_model as load_rec_model
    from surya.model.recognition.processor import load_processor as load_rec_processor

    import torch
    _n = torch.cuda.device_count()
    print(
        f'[worker] CUDA_VISIBLE_DEVICES={os.environ.get("CUDA_VISIBLE_DEVICES")!r} '
        f'| surya device={settings.TORCH_DEVICE_MODEL} | torch sees {_n} gpu '
        f'| {torch.cuda.get_device_name(0) if _n > 0 else "NO GPU"}',
        flush=True,
    )

    # Nâng độ phân giải crop chữ (mặc định height 256 / width 896). Bỏ qua nếu bản
    # surya không cho gán field (pydantic frozen) - khi đó giữ mặc định.
    try:
        settings.RECOGNITION_IMAGE_SIZE = {'height': REC_IMG_HEIGHT, 'width': 896}
    except Exception as e:
        print(f'[WARN] khong set RECOGNITION_IMAGE_SIZE={REC_IMG_HEIGHT}: {type(e).__name__}: {e}', flush=True)

    det_model = load_det_model()
    det_processor = load_det_processor()
    rec_model = load_rec_model()
    rec_processor = load_rec_processor()
    return (run_ocr, det_model, det_processor, rec_model, rec_processor)


def _img_text(pred):
    out = []
    for line in getattr(pred, 'text_lines', []) or []:
        t = (line.text or '').strip()
        if not t:
            continue
        conf = getattr(line, 'confidence', None)
        if conf is not None and float(conf) < MIN_CONF:
            continue
        out.append(t)
    return ' '.join(out)


def ocr_paths(paths, engine):
    from PIL import Image
    run_ocr, det_model, det_processor, rec_model, rec_processor = engine
    imgs = [Image.open(p).convert('RGB') for p in paths]
    preds = run_ocr(imgs, [LANGS] * len(imgs), det_model, det_processor,
                    rec_model, rec_processor)
    return [_img_text(p) for p in preds]


def ocr_one(path, engine):
    return ocr_paths([path], engine)[0]


def out_path(video_name):
    return Path(OUTPUT_DIR) / f'{video_name}.json'


def load_video_ocr(video_name):
    p = out_path(video_name)
    if p.exists():
        with p.open(encoding='utf-8') as f:
            return json.load(f)
    return {}


def save_video_ocr(video_name, data):
    with out_path(video_name).open('w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False)


def build_video_index():
    root = Path(KEYFRAMES_DIR)
    videos = {}
    for img in root.rglob('*'):
        if img.is_file() and img.suffix.lower() in IMAGE_EXTS:
            video_name = img.parent.name
            frame_id = f'{video_name}_{img.stem}'
            videos.setdefault(video_name, []).append((frame_id, img))
    for v in videos:
        videos[v].sort(key=lambda t: t[0])
    return videos


def run_smoke():
    first = next(
        p for p in sorted(Path(KEYFRAMES_DIR).rglob('*'))
        if p.suffix.lower() in IMAGE_EXTS
    )
    print('smoke frame:', first, flush=True)
    engine = init_engine()
    t0 = time.time()
    text = ocr_one(first, engine)
    print(f'inference: {time.time() - t0:.3f}s', flush=True)
    print('OCR ->', repr(text[:200]), flush=True)


def run_corpus(video_names):
    engine = init_engine()
    print(f'engine ready: langs={LANGS} rec_img_h={REC_IMG_HEIGHT}', flush=True)
    videos = build_video_index()
    total_done = 0
    t_start = time.time()
    for video_name in video_names:
        frames = videos[video_name][::FRAME_STRIDE]
        data = load_video_ocr(video_name)
        todo = [(fid, p) for fid, p in frames if fid not in data]
        if not todo:
            total_done += len(frames)
            continue
        done_in_video = 0
        for i in range(0, len(todo), IMG_BATCH):
            chunk = todo[i:i + IMG_BATCH]
            paths = [p for _, p in chunk]
            try:
                texts = ocr_paths(paths, engine)
            except Exception as e:
                print(f'[WARN] batch failed: {type(e).__name__}: {e}', flush=True)
                texts = [ocr_one(p, engine) for _, p in chunk]
            for (fid, _p), t in zip(chunk, texts):
                data[fid] = t
            done_in_video += len(chunk)
            if done_in_video % CHECKPOINT_EVERY == 0:
                save_video_ocr(video_name, data)
        save_video_ocr(video_name, data)
        total_done += len(frames)
        print(
            f'{video_name}: {len(frames)} frames | '
            f'cum {total_done} | {time.time() - t_start:.1f}s',
            flush=True,
        )
    print(f'done {total_done} frames in {(time.time() - t_start) / 60:.1f} min', flush=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--gpu', type=int, default=-1)
    ap.add_argument('--videos', nargs='*')
    ap.add_argument('--smoke', action='store_true')
    args = ap.parse_args()
    # Pin worker vào đúng GPU TRƯỚC khi import surya/torch để auto-detect dùng đúng card.
    if args.gpu >= 0:
        os.environ['CUDA_VISIBLE_DEVICES'] = str(args.gpu)
    if args.smoke:
        run_smoke()
    else:
        run_corpus(sorted(args.videos))


if __name__ == '__main__':
    main()
'''
with open(WORKER_SCRIPT, 'w', encoding='utf-8') as f:
    f.write(WORKER_SRC)
print('worker script ->', WORKER_SCRIPT)


In [ ]:
# Smoke test qua worker trên GPU 0 - sanity check trước khi chạy cả corpus.
# (Lần đầu sẽ download model det+rec từ HF Hub, hơi lâu.)
cmd = [sys.executable, WORKER_SCRIPT, '--gpu', '0', '--smoke']
r = subprocess.run(cmd, capture_output=True, text=True,
                   encoding='utf-8', errors='replace')
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise SystemExit('smoke test failed')


In [ ]:
# Group mọi keyframe theo video: {video_name: [(frame_id, path), ...]}
root = Path(KEYFRAMES_DIR)
videos = {}
for img in root.rglob('*'):
    if img.is_file() and img.suffix.lower() in IMAGE_EXTS:
        video_name = img.parent.name                  # e.g. L21_V001
        frame_id   = f'{video_name}_{img.stem}'       # img.stem = zero-padded idx
        videos.setdefault(video_name, []).append((frame_id, img))
for v in videos:
    videos[v].sort(key=lambda t: t[0])

print('videos:', len(videos),
      '| frames:', sum(len(v) for v in videos.values()),
      '| stride:', FRAME_STRIDE)


In [ ]:
# Chia video round-robin cho NUM_GPUS worker, chạy 2 tiến trình song song.
video_names = sorted(videos)
chunks = [video_names[i::NUM_GPUS] for i in range(NUM_GPUS)]
for g in range(NUM_GPUS):
    print(f'gpu{g}: {len(chunks[g])} videos')

t_start = time.time()
procs = []
for gpu, vlist in enumerate(chunks):
    if not vlist:
        continue
    cmd = [sys.executable, WORKER_SCRIPT, '--gpu', str(gpu), '--videos', *vlist]
    p = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        encoding='utf-8',
        errors='replace',
    )
    procs.append(p)

# Snapshot nvidia-smi ngay sau khi launch - xem session thực sự có mấy GPU.
print('--- nvidia-smi snapshot ---')
_smi = subprocess.run(
    ['nvidia-smi', '--query-gpu=index,name,utilization.gpu,memory.used', '--format=csv'],
    capture_output=True, text=True)
print(_smi.stdout or _smi.stderr)

# Stream output 2 worker live rồi đợi cả 2 xong.
for p in procs:
    for line in p.stdout:
        print(line, end='', flush=True)
    rc = p.wait()
    if rc != 0:
        raise SystemExit(f'worker exited with code {rc}')
elapsed = time.time() - t_start
print(f'\nall workers done in {elapsed/60:.1f} min')


In [ ]:
# Merge các file per-video thành 1 file flat đúng format backend.
# (Khi copy về local: file này sẽ thay thẳng backend/data/ocr/ocr_data.json.)
merged = {}
for jf in sorted(Path(OUTPUT_DIR).glob('*.json')):
    with jf.open(encoding='utf-8') as f:
        merged.update(json.load(f))

with open('ocr_data.json', 'w', encoding='utf-8') as f:
    json.dump(merged, f, ensure_ascii=False)

print(f'merged entries: {len(merged)} -> ocr_data.json')
print(f'non-empty:      {sum(1 for v in merged.values() if v)} '
      f'({sum(1 for v in merged.values() if v)/max(len(merged),1)*100:.1f}%)')


## Dùng với backend

Sau khi notebook chạy xong, download `ocr_data.json` về và copy đúng chỗ:

```bash
# 1) OCR mới thay thẳng file Qwen/EasyOCR cũ (drop-in, format giống hệt)
cp ocr_data.json  backend/data/ocr/ocr_data.json

# 2) Rebuild lại các index phụ thuộc OCR (scene text trong graph cũng đổi):
cd data_processing
python 07_build_text_indexes.py     # ocr_text + asr_text + caption_text index
python 08_build_graph.py            # scene.ocr lấy từ OCR mới
```

## Nâng cấp tuỳ chọn

1. **Chữ banner nhỏ bị miss**: tăng `REC_IMG_HEIGHT` lên 384/512 (chậm hơn ~2x).
2. **Nhanh hơn**: giảm `REC_IMG_HEIGHT` về 256 hoặc tăng `IMG_BATCH`.
3. Nếu vẫn cần chất lượng Qwen cho scene khó, chạy song song: Surya/PP-OCRv5 phủ nhanh,
   Qwen chỉ cho các frame Surya cho kết quả rỗng/đáng ngờ.
